<a href="https://colab.research.google.com/github/Somaskandan931/flyrank-ml-2026-Somaskandan931/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Somaskandan931/flyrank-ml-2026-Somaskandan931/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip install -q duckdb huggingface_hub pandas scikit-learn

from huggingface_hub import login
from google.colab import userdata
import duckdb, pandas as pd, numpy as np, json, os

login(token=userdata.get('HF_TOKEN'))

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("CREATE SECRET (TYPE huggingface, TOKEN '" + userdata.get('HF_TOKEN') + "')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT_DAILY = f"{BASE}/fact_content_daily_performance/**/*.parquet"

In [ ]:
# ============ CONFIG ============
GROUP_COL = "client_hash_id"          # matches your ML-04/07 naming — used for the grouped split, never as a feature
LABEL_COL = "label_high_clicks_late"  # same binary label defined in ML-04
ID_COLS = ["content_hash_id"]
FEATURE_COLS = [
    "gsc_impressions_early", "gsc_clicks_early",
    "gsc_avg_position_early", "ga4_sessions_early", "scroll_events_early",
]
K_FOR_PRECISION = 50    # same K as your Top-20/queue-style ranking in ML-07, adjust if you used a different K there
RANDOM_STATE = 42
EARLY_START, EARLY_END = "2026-03-01", "2026-03-15"
LATE_START, LATE_END = "2026-03-16", "2026-03-31"

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My ML-04 data contract already separates an early window (features) from a late window
(label: whether a page's clicks in the back half of March exceed the month's median). That's
a genuine binary classification task, so I'm using **Random Forest** as the primary model,
with **Logistic Regression** (the same model family my ML-04 leakage check used) and a
shallow **Decision Tree** as counterparts on the same split and metric. A tree ensemble can
pick up interactions the honest logistic model in ML-04 (AUC 0.86) can't — e.g. high
impressions only predicting future clicks when position is already decent — and it gives
me feature importances for Section 4. I'm adding Gradient Boosting as the optional stretch.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    precision_score, recall_score, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
)

# Rebuild the same early/late feature table from ML-04 (never persisted to disk, so
# it's reconstructed here directly from the warehouse — same query, same windows)
df = con.sql(f"""
WITH early AS (
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS gsc_impressions_early,
           SUM(gsc_clicks)      AS gsc_clicks_early,
           SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS gsc_avg_position_early,
           SUM(ga4_sessions)    AS ga4_sessions_early,
           SUM(scroll_events)   AS scroll_events_early
    FROM read_parquet('{FACT_DAILY}')
    WHERE month = '2026-03'
      AND report_date BETWEEN '{EARLY_START}' AND '{EARLY_END}'
      AND gsc_data_available IS TRUE
    GROUP BY 1,2
),
late AS (
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_clicks) AS gsc_clicks_late
    FROM read_parquet('{FACT_DAILY}')
    WHERE month = '2026-03'
      AND report_date BETWEEN '{LATE_START}' AND '{LATE_END}'
      AND gsc_data_available IS TRUE
    GROUP BY 1,2
)
SELECT e.*, l.gsc_clicks_late
FROM early e
JOIN late l USING (content_hash_id, client_hash_id)
""").df()

df = df.dropna(subset=FEATURE_COLS + ["gsc_clicks_late"])
median_clicks = df["gsc_clicks_late"].median()
df[LABEL_COL] = (df["gsc_clicks_late"] > median_clicks).astype(int)

print(f"Loaded {df.shape[0]} rows, label positive rate: {df[LABEL_COL].mean():.3f}")
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 75222 rows, label positive rate: 0.372


,content_hash_id,client_hash_id,gsc_impressions_early,gsc_clicks_early,gsc_avg_position_early,ga4_sessions_early,scroll_events_early,gsc_clicks_late,label_high_clicks_late
4117,content_60c47a2bd592c26e,client_65de48885f4ef01b,11.0,0.0,35.454545,0.0,0.0,0.0,0
4118,content_db76968a5edaa234,client_65de48885f4ef01b,7.0,0.0,3.428571,0.0,0.0,0.0,0
4119,content_45a68353d4d5740e,client_65de48885f4ef01b,21.0,0.0,45.857143,0.0,0.0,0.0,0
4120,content_e97b6b92a4143bc1,client_65de48885f4ef01b,38.0,0.0,62.973684,0.0,0.0,0.0,0
4121,content_529583c7206b4164,client_65de48885f4ef01b,49.0,0.0,31.326531,2.0,0.0,0.0,0


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

My ML-04 notebook used a plain `train_test_split` with no grouping — honest for the
leakage demo it was doing, but not honest enough for a real model comparison, since rows
from the same client share systemic properties (site size, niche) that a random split
would let leak between train and test. Here I switch to a **client-grouped held-out split**
(`GroupShuffleSplit` on `client_hash_id`), so no client appears on both sides. The
early/late time separation from ML-04 is preserved (features only ever come from
March 1–15, the label only from March 16–31) — the grouped split adds a second, orthogonal
layer of honesty on top of that time-aware design, not a replacement for it.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(df, groups=df[GROUP_COL]))
train_df, test_df = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

overlap = set(train_df[GROUP_COL]) & set(test_df[GROUP_COL])
assert not overlap, f"Client leakage across split: {len(overlap)} shared clients"

print(f"Train: {len(train_df)} rows / {train_df[GROUP_COL].nunique()} clients")
print(f"Test:  {len(test_df)} rows / {test_df[GROUP_COL].nunique()} clients")

Train: 58027 rows / 26 clients
Test:  17195 rows / 7 clients


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

My Week-4 (ML-07) baseline — CTR Recovery Priority — was built on the *whole* March month,
not the early/late split this task needs. To compare it fairly against the trained models
below, I recompute the same rule using only the early-window columns (`gsc_impressions_early`,
`gsc_clicks_early`, `gsc_avg_position_early`), with the position-band benchmark CTR fit on
the train split only (never touching test), then score the held-out test rows and rank them
exactly like the trained models — same features, same label, same split, same Precision@K.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
X_train, y_train = train_df[FEATURE_COLS], train_df[LABEL_COL]
X_test, y_test = test_df[FEATURE_COLS], test_df[LABEL_COL]

def precision_at_k(y_true, scores, k):
    order = np.argsort(scores)[::-1][:k]
    y_true = np.asarray(y_true)
    return y_true[order].mean()

# --- Baseline: CTR Recovery Priority, recomputed on the early window, benchmark fit on train only ---
def position_bucket(pos):
    if pos <= 3: return "1-3"
    if pos <= 10: return "4-10"
    if pos <= 20: return "11-20"
    if pos <= 50: return "21-50"
    return "51+"

train_df["position_bucket"] = train_df["gsc_avg_position_early"].apply(position_bucket)
test_df["position_bucket"] = test_df["gsc_avg_position_early"].apply(position_bucket)

benchmark_ctr = (
    train_df.assign(ctr=train_df["gsc_clicks_early"] / train_df["gsc_impressions_early"])
    .groupby("position_bucket")
    .apply(lambda g: g["gsc_clicks_early"].sum() / g["gsc_impressions_early"].sum())
)

test_df["ctr_early"] = test_df["gsc_clicks_early"] / test_df["gsc_impressions_early"]
test_df["benchmark_ctr"] = test_df["position_bucket"].map(benchmark_ctr)
test_df["baseline_score"] = ((test_df["benchmark_ctr"] - test_df["ctr_early"]).clip(lower=0)
                              * test_df["gsc_impressions_early"])

baseline_row = {
    "method": "baseline_ctr_recovery_week4",
    f"precision_at_{K_FOR_PRECISION}": round(precision_at_k(y_test, test_df["baseline_score"].values, K_FOR_PRECISION), 3),
    "roc_auc": round(roc_auc_score(y_test, test_df["baseline_score"]), 3),
    "avg_precision": round(average_precision_score(y_test, test_df["baseline_score"]), 3),
    "precision": np.nan,   # rule has no fixed 0.5 threshold, so precision/recall at threshold aren't defined
    "recall": np.nan,
}

# --- Trained models ---
models = {
    "logistic_regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "decision_tree": DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE, class_weight="balanced"),
    "random_forest": RandomForestClassifier(
        n_estimators=300, max_depth=8, random_state=RANDOM_STATE, class_weight="balanced", n_jobs=-1
    ),
    "gradient_boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

fitted = {}
rows = [baseline_row]
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    scores = model.predict_proba(X_test)[:, 1]
    preds = (scores >= 0.5).astype(int)
    rows.append({
        "method": name,
        f"precision_at_{K_FOR_PRECISION}": round(precision_at_k(y_test, scores, K_FOR_PRECISION), 3),
        "precision": round(precision_score(y_test, preds, zero_division=0), 3),
        "recall": round(recall_score(y_test, preds, zero_division=0), 3),
        "roc_auc": round(roc_auc_score(y_test, scores), 3),
        "avg_precision": round(average_precision_score(y_test, scores), 3),
    })

results_df = pd.DataFrame(rows).set_index("method")
results_df

/tmp/ipykernel_4108/1424910798.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g["gsc_clicks_early"].sum() / g["gsc_impressions_early"].sum())


,precision_at_50,roc_auc,avg_precision,precision,recall
method,,,,,
baseline_ctr_recovery_week4,0.94,0.567,0.554,NaN,NaN
logistic_regression,1.00,0.871,0.825,0.825,0.610
decision_tree,0.94,0.877,0.817,0.705,0.774
random_forest,1.00,0.882,0.832,0.720,0.756
gradient_boosting,1.00,0.884,0.834,0.797,0.650


**Observed:** fill this in once the table above runs — e.g. "the random forest lifts
Precision@50 over the recomputed baseline rule by ~Nx on held-out clients, using the same
early-window features, label, and split." Keep it to what the numbers show — observed /
measured / directional, not a causal or predictive claim about future search performance.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
best_name = results_df.drop(index=["baseline_ctr_recovery_week4"])[f"precision_at_{K_FOR_PRECISION}"].idxmax()
best_model = fitted[best_name]
print(f"Interpreting: {best_name}")

test_scored = test_df.copy()
test_scored["score"] = best_model.predict_proba(X_test)[:, 1]
test_scored["pred"] = (test_scored["score"] >= 0.5).astype(int)

cm = confusion_matrix(y_test, test_scored["pred"])
print("Confusion matrix [ [TN FP] / [FN TP] ]:\n", cm)
print()
print(classification_report(y_test, test_scored["pred"], zero_division=0))

if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
elif hasattr(best_model, "coef_"):
    importances = pd.Series(best_model.coef_[0], index=FEATURE_COLS).abs().sort_values(ascending=False)
print("\nFeature importances:")
print(importances)

worst_fp = test_scored[test_scored[LABEL_COL] == 0].sort_values("score", ascending=False).head(10)
worst_fn = test_scored[test_scored[LABEL_COL] == 1].sort_values("score", ascending=True).head(10)

display_cols = [GROUP_COL, "score", LABEL_COL] + FEATURE_COLS
print("\nWorst false positives (predicted high-clicks-late, actually below median):")
print(worst_fp[display_cols])
print("\nWorst false negatives (predicted low, actually above median):")
print(worst_fn[display_cols])

Interpreting: logistic_regression
Confusion matrix [ [TN FP] / [FN TP] ]:
 [[10305   791]
 [ 2379  3720]]

              precision    recall  f1-score   support

           0       0.81      0.93      0.87     11096
           1       0.82      0.61      0.70      6099

    accuracy                           0.82     17195
   macro avg       0.82      0.77      0.78     17195
weighted avg       0.82      0.82      0.81     17195


Feature importances:
gsc_clicks_early          0.741642
scroll_events_early       0.015127
gsc_avg_position_early    0.013715
ga4_sessions_early        0.001586
gsc_impressions_early     0.000606
dtype: float64

Worst false positives (predicted high-clicks-late, actually below median):
                 client_hash_id     score  label_high_clicks_late  \
123657  client_3f0ce4d44fe94f3d  0.999999                       0   
50495   client_3f0ce4d44fe94f3d  0.999970                       0   
102675  client_fef1a8f436438636  0.999918                       0   
95

Fill in once the cell above runs: which 2-3 early-window features the model leans on most
(e.g. does it echo ML-06's signal audit, or surface something new like scroll depth), and
what the worst false positives/negatives have in common — e.g. "false positives are early-window
high-impression pages that just didn't convert late-window traffic, similar to the noisy
small-sample rows flagged in ML-07's weak-picks check." Observed / measured / directional —
no causal claim about why a page's late-window performance moved.

## Self-check

Before you submit, confirm each line honestly:
- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [ ]:
os.makedirs("work/outputs", exist_ok=True)
results_df.reset_index().to_json("work/outputs/w05_model_metrics.json", orient="records", indent=2)
print("Wrote work/outputs/w05_model_metrics.json")

Wrote work/outputs/w05_model_metrics.json
